# TranSTR + DINOv3 + GroundingDINO-SigLIP2 + Top-5 Temporal Relation + NCOD LUM+HUM — Colab Pro
**Object feature mới**: SigLIP2 ROI 768D + DeBERTa class 768D + bbox 4D = 1540D. Data tải từ Kaggle; bs=32, không gradient accumulation, 20 epochs, EMA weights cho eval/save.


In [ ]:
# CELL 1: Git Clone & Setup (Google Colab)
import os
from pathlib import Path

REPO_URL = 'https://github.com/DanielQH07/tranSTR_Casual.git'
REPO_NAME = 'tranSTR_Casual'
BRANCH = 'improve'
WORKING_DIR = Path('/content')
REPO_DIR = WORKING_DIR / REPO_NAME

def has_repo_files(p):
    p = Path(p)
    return (p / 'DataLoader.py').exists() and (p / 'networks' / 'model.py').exists()

if not has_repo_files(Path.cwd()):
    if not REPO_DIR.exists():
        rc = os.system(f'git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}')
        if rc != 0:
            raise RuntimeError('git clone failed')
    target_dir = REPO_DIR / 'causalvid'
    os.chdir(target_dir if target_dir.exists() else REPO_DIR)
if not has_repo_files(Path.cwd()):
    raise FileNotFoundError(f'Repo files not found in {Path.cwd()}')
print(f'Working directory: {Path.cwd()}')


Working directory: /content/tranSTR_Casual


In [ ]:
%cd /content/tranSTR_Casual

/content


In [ ]:
# CELL 2: Dependencies + W&B login (Colab)
print('=== CELL 2: Dependencies & W&B Setup ===')
import os, sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'wandb', 'kagglehub', 'transformers', 'sentencepiece'])
import wandb

WANDB_PROJECT = 'transtr-causalvid-dino'
WANDB_ENTITY = "haidang262004-i-h-c-qu-c-gia-tp-hcm"
wandb_key = ""
try:
    from google.colab import userdata
    wandb_key = wandb_key or (userdata.get('WANDB_API_KEY') or '').strip()
except Exception:
    pass
if wandb_key:
    wandb.login(key=wandb_key)
    print('W&B login OK')
else:
    os.environ.setdefault('WANDB_MODE', 'offline')
    print('WANDB_API_KEY is not configured; W&B offline mode enabled')


=== CELL 2: Dependencies & W&B Setup ===


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


W&B login OK


In [ ]:
# CELL 3: Resolve data paths from KaggleHub (Colab)
print('=== CELL 3: Data Paths (KaggleHub) ===')
import os, shutil
from pathlib import Path
import kagglehub

def _download(slug, env_name=None):
    override = os.environ.get(env_name, '').strip() if env_name else ''
    return Path(override) if override and Path(override).exists() else Path(kagglehub.dataset_download(slug))
def _find_dir_with_ext(root, ext):
    counts = {}
    for p in Path(root).rglob('*' + ext):
        counts[str(p.parent)] = counts.get(str(p.parent), 0) + 1
    return Path(max(counts, key=counts.get)) if counts else None
def _find_named(root, name):
    root = Path(root)
    if root.name.lower() == name.lower():
        return root
    return next((p for p in root.rglob('*') if p.is_dir() and p.name.lower() == name.lower()), None)

dinov3_root = _download('danielq07/dinov3-feat', 'DINOV3_DATASET_PATH')
gdino_root = _download(os.environ.get('GDINO_DATASET', 'danielq07/causal-vidqa-gdinofasterrcnn-features-merged'), 'GDINO_DATASET_PATH')
anno_root = _download('lusnaw/text-annotation', 'ANNO_DATASET_PATH')
split_root = _download('danielq07/casual-vid-data-split', 'SPLIT_DATASET_PATH')

all_dinov3_pt = list(Path(dinov3_root).rglob('*.pt'))
if not all_dinov3_pt:
    raise FileNotFoundError(f'No DINOv3 .pt files found under {dinov3_root}')

GDINO_MERGED_PATH = _find_dir_with_ext(gdino_root, '.pkl') or gdino_root
ANNOTATION_QA_PATH = _find_named(anno_root, 'QA') or anno_root
SPLIT_TXT_PATH = _find_named(split_root, 'split') or split_root

# DataLoader expects one flat directory. KaggleHub/Colab cache keeps DINOv3
# under train/valid/test, so link all three splits into a single view.
CLIP_MERGED_PATH = Path('/content/dinov3_T16_dim1024_merge')
CLIP_MERGED_PATH.mkdir(parents=True, exist_ok=True)
for src in all_dinov3_pt:
    dst = CLIP_MERGED_PATH / src.name
    if dst.exists():
        continue
    try:
        dst.symlink_to(src)
    except Exception:
        shutil.copy2(src, dst)

merged_count = len(list(CLIP_MERGED_PATH.glob('*.pt')))
source_unique_count = len({src.name for src in all_dinov3_pt})
if merged_count != source_unique_count:
    raise RuntimeError(
        f'DINOv3 merge incomplete: merged={merged_count}, source_unique={source_unique_count}'
    )
print(f'DINOv3 merged all splits: {merged_count} files')
print('Resolved paths:', CLIP_MERGED_PATH, GDINO_MERGED_PATH, ANNOTATION_QA_PATH, SPLIT_TXT_PATH)
for name, path in [('DINOv3', CLIP_MERGED_PATH), ('GDINO+FRCNN', GDINO_MERGED_PATH), ('QA', ANNOTATION_QA_PATH), ('split', SPLIT_TXT_PATH)]:
    if not Path(path).exists(): raise FileNotFoundError(f'{name} path not found: {path}')


=== CELL 3: Data Paths (KaggleHub) ===
Using Colab cache for faster access to the 'dinov3-feat' dataset.
Using Colab cache for faster access to the 'casual-vid-data-split' dataset.
DINOv3 merged all splits: 26900 files
Resolved paths: /content/dinov3_T16_dim1024_merge /root/.cache/kagglehub/datasets/danielq07/causal-vidqa-gdinofasterrcnn-features-merged/versions/1/gdinofasterrcnn_features /root/.cache/kagglehub/datasets/lusnaw/text-annotation/versions/2/QA /kaggle/input/casual-vid-data-split/split


In [ ]:
# CELL 4: Core imports + NCOD/HUM + Verifier/Knowledge training functions
print('=== CELL 4: Imports + Functions (NCOD + HUM + Verifier/Knowledge) ===')

import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from IPython.display import display

from utils.util import set_seed, set_gpu_devices
from DataLoader import VideoQADataset
from networks.model import VideoQAmodel

def _unpack_batch(batch):
    if len(batch) == 7:
        ff, of, q, a, ans_id, qns_key, q_family_id = batch
    elif len(batch) == 6:
        ff, of, q, a, ans_id, qns_key = batch
        q_family_id = None
    else:
        raise ValueError(f'Unexpected batch format with {len(batch)} elements')
    return ff, of, q, a, ans_id, qns_key, q_family_id

def _compute_sample_indices(qns_keys, key_to_idx, device):
    idxs = [key_to_idx.get(str(k), -1) for k in qns_keys]
    if any(i < 0 for i in idxs):
        missing = [str(qns_keys[i]) for i, v in enumerate(idxs) if v < 0][:5]
        raise KeyError(f'Missing qns_key in key_to_idx mapping: {missing}')
    return torch.tensor(idxs, dtype=torch.long, device=device)

_QTYPE_SUFFIXES = [
    'counterfactual_reason',
    'predictive_reason',
    'counterfactual',
    'predictive',
    'explanatory',
    'descriptive',
]

def _split_qns_key(qns_key):
    key = str(qns_key)
    for qtype in _QTYPE_SUFFIXES:
        suffix = f'_{qtype}'
        if key.endswith(suffix):
            return key[:-len(suffix)], qtype
    return key, 'unknown'

def _compute_acc_all_metrics(type_results):
    mapping = [
        ('Description', 'descriptive'),
        ('Explanation', 'explanatory'),
        ('Predictive-Answer', 'predictive'),
        ('Predictive-Reason', 'predictive_reason'),
        ('Counterfactual-Answer', 'counterfactual'),
        ('Counterfactual-Reason', 'counterfactual_reason'),
    ]
    metrics = {}
    for name, qtype in mapping:
        rows = type_results.get(qtype, [])
        metrics[name] = (sum(1 for r in rows if r['correct']) / len(rows) * 100) if rows else 0.0

    def _hard_metric(type_ans, type_reason):
        ans_by_vid = {r['video_id']: r['correct'] for r in type_results.get(type_ans, [])}
        reason_by_vid = {r['video_id']: r['correct'] for r in type_results.get(type_reason, [])}
        common_vids = set(ans_by_vid) & set(reason_by_vid)
        if not common_vids:
            return 0.0
        both_correct = sum(1 for vid in common_vids if ans_by_vid[vid] and reason_by_vid[vid])
        return both_correct / len(common_vids) * 100

    metrics['PAR'] = _hard_metric('predictive', 'predictive_reason')
    metrics['CAR'] = _hard_metric('counterfactual', 'counterfactual_reason')
    metrics['Acc_ALL'] = (metrics['Description'] + metrics['Explanation'] + metrics['PAR'] + metrics['CAR']) / 4.0
    return metrics

def _hard_negative_weights(cand_feat, target, enabled=False, max_weight=1.5):
    if (not enabled) or cand_feat is None or max_weight <= 1.0:
        return torch.ones_like(target, dtype=torch.float32)
    with torch.no_grad():
        cand_norm = F.normalize(cand_feat.detach(), dim=-1)
        gold = cand_norm.gather(1, target.view(-1, 1, 1).expand(-1, 1, cand_norm.size(-1))).squeeze(1)
        sims = torch.bmm(cand_norm, gold.unsqueeze(-1)).squeeze(-1)
        sims.scatter_(1, target.view(-1, 1), -1.0)
        hardness = sims.max(dim=1).values.clamp(min=0.0, max=1.0)
        return 1.0 + (float(max_weight) - 1.0) * hardness

def _update_ema_model(ema_model, model, decay):
    if ema_model is None:
        return
    with torch.no_grad():
        for ema_param, param in zip(ema_model.parameters(), model.parameters()):
            ema_param.data.mul_(decay).add_(param.data, alpha=1.0 - decay)
        for ema_buffer, buffer in zip(ema_model.buffers(), model.buffers()):
            ema_buffer.copy_(buffer)

def train_epoch_integrated(
    model, optimizer_model, optimizer_u, U, loader, xe, bce, device, epoch, key_to_idx,
    accumulation_steps=4, hum_alpha=1.0, lambda_verifier=0.2, lambda_knowledge=0.1,
    use_hard_neg_mining=False, hard_neg_weight_max=1.5,
    ema_model=None, ema_decay=0.999,
    scheduler=None, scheduler_step_per_batch=False
):
    model.train()
    total_loss, total_l1, total_l2 = 0.0, 0.0, 0.0
    total_verifier, total_knowledge = 0.0, 0.0
    correct, total = 0, 0
    optimizer_model.zero_grad()
    optimizer_u.zero_grad()

    pbar = tqdm(loader, desc=f'Epoch {epoch}', leave=False)
    for batch_idx, batch in enumerate(pbar):
        ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
        ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)

        if q_family_id is None:
            q_family_id = torch.zeros_like(tgt)
        else:
            q_family_id = q_family_id.to(device)

        sample_indices = _compute_sample_indices(qns_keys, key_to_idx, device)
        out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
        logits = out['logits']
        fused_logits = out.get('fused_score', logits)
        verifier_logits = out.get('verifier_logits', logits)
        knowledge_logits = out.get('knowledge_score', None)
        cand_feat = out.get('cand_feat', None)

        probs = torch.softmax(logits, dim=1)
        y_onehot = F.one_hot(tgt, num_classes=logits.size(-1)).float()
        u_batch = U[sample_indices].unsqueeze(1)

        ce_per_sample = -torch.sum(y_onehot * torch.log(torch.clamp(probs, min=1e-12)), dim=1)
        shifted_probs = torch.clamp(probs + (u_batch.detach() * y_onehot), min=1e-12, max=1.0)
        lum_loss = -torch.sum(y_onehot * torch.log(shifted_probs), dim=1)
        hum_loss = (1.0 + hum_alpha * u_batch.detach().squeeze(1)) * ce_per_sample

        hard_neg_w = _hard_negative_weights(
            cand_feat,
            tgt,
            enabled=use_hard_neg_mining,
            max_weight=hard_neg_weight_max,
        )
        hard_mask = (q_family_id >= 2)
        l1_per_sample = torch.where(hard_mask, hum_loss, lum_loss)
        l1 = (l1_per_sample * hard_neg_w).mean()

        verifier_loss = bce(verifier_logits, y_onehot)
        if lambda_knowledge > 0.0 and knowledge_logits is not None:
            knowledge_loss = bce(knowledge_logits, y_onehot)
        else:
            knowledge_loss = torch.tensor(0.0, device=device)

        model_loss = l1 + lambda_verifier * verifier_loss + lambda_knowledge * knowledge_loss
        (model_loss / accumulation_steps).backward()

        shifted_det = probs.detach() + (u_batch * y_onehot)
        l2 = F.mse_loss(shifted_det, y_onehot)
        (l2 / accumulation_steps).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer_model.step()
            _update_ema_model(ema_model, model, ema_decay)
            if scheduler is not None and scheduler_step_per_batch:
                scheduler.step()
            optimizer_model.zero_grad()
            optimizer_u.step()
            optimizer_u.zero_grad()
            with torch.no_grad():
                U.clamp_(0.0, 0.99)

        total_l1 += l1.item()
        total_l2 += l2.item()
        total_verifier += verifier_loss.item()
        total_knowledge += knowledge_loss.item()
        total_loss += (model_loss + l2).item()
        correct += (fused_logits.argmax(-1) == tgt).sum().item()
        total += tgt.size(0)

        pbar.set_postfix({
            'loss': total_loss / (batch_idx + 1),
            'l1': total_l1 / (batch_idx + 1),
            'l2': total_l2 / (batch_idx + 1),
            'ver': total_verifier / (batch_idx + 1),
            'know': total_knowledge / (batch_idx + 1),
            'acc': correct / max(total, 1) * 100
        })

    n = len(loader)
    return (
        total_loss / n,
        total_l1 / n,
        total_l2 / n,
        total_verifier / n,
        total_knowledge / n,
        correct / max(total, 1) * 100
    )

def eval_epoch(model, loader, device):
    model.eval()
    correct, total = 0, 0
    type_results = {}
    with torch.no_grad():
        for batch in loader:
            ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            preds = logits.argmax(-1)
            correct += (preds == tgt).sum().item()
            total += tgt.size(0)

            for key, pred, target in zip(qns_keys, preds.detach().cpu().tolist(), tgt.detach().cpu().tolist()):
                video_id, qtype = _split_qns_key(key)
                type_results.setdefault(qtype, []).append({
                    'video_id': video_id,
                    'correct': bool(int(pred) == int(target)),
                })

    metrics = _compute_acc_all_metrics(type_results)
    metrics['Plain_Acc'] = correct / max(total, 1) * 100
    return metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Imports and functions defined for integrated training.')

=== CELL 4: Imports + Functions (NCOD + HUM + Verifier/Knowledge) ===
Device: cuda
GPU: Tesla T4 | VRAM: 15.6 GB
Imports and functions defined for integrated training.


In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Imports OK')

Device: cuda
GPU: Tesla T4 | VRAM: 15.6 GB
Imports OK


In [ ]:
# CELL 5: Setup Paths & Config (Kaggle, bs=8 + accum=4 -> effective bs=32)
print('=== CELL 5: Paths & Config ===')

clip_merged_path   = globals().get('CLIP_MERGED_PATH', None)
gdino_merged_path  = globals().get('GDINO_MERGED_PATH', None)
annotation_qa_path = globals().get('ANNOTATION_QA_PATH', None)
split_txt_path     = globals().get('SPLIT_TXT_PATH', None)

CLIP_FEATURE_PATH  = clip_merged_path  or '/content/dinov3_T16_dim1024_merge'
GDINO_FEATURE_PATH = gdino_merged_path or str(GDINO_MERGED_PATH)
ANNOTATION_PATH    = annotation_qa_path or str(ANNOTATION_QA_PATH)
SPLIT_DIR          = split_txt_path    or str(SPLIT_TXT_PATH)

BASE = '/content/working' if os.path.exists('/content') else os.getcwd()
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print('\n--- Path Verification ---')
def verify_path(name, path):
    if os.path.exists(path):
        items = os.listdir(path)[:3]
        print(f'OK {name}: {items}')
        return True
    print(f'NOT FOUND {name}: {path}')
    return False

all_ok = True
all_ok &= verify_path('DINOv3 Features (1024D)', CLIP_FEATURE_PATH)
all_ok &= verify_path('GDINO+FRCNN Features (2820D)', GDINO_FEATURE_PATH)
all_ok &= verify_path('Annotations (QA)', ANNOTATION_PATH)
all_ok &= verify_path('Splits', SPLIT_DIR)

if not all_ok:
    raise FileNotFoundError('One or more required data paths are missing. Re-run CELL 3 or attach Kaggle datasets.')

import glob as _glob
n_pt  = len(_glob.glob(os.path.join(CLIP_FEATURE_PATH, '*.pt')))
n_pkl = len(_glob.glob(os.path.join(GDINO_FEATURE_PATH, '*.pkl')))
print(f'\nDINOv3 .pt: {n_pt} | GDINO+FRCNN .pkl: {n_pkl}')

# ============================================
# 3-RUN TUNING PRESETS
# ============================================
RUN_TRAINING = True
RUN_PROFILE = 'run1'
RUN_VARIANT = 'selectivetopk_splitgate_mask_v2_nokb'
RUN3_REG_MODE = 'dropout'

RUN_PROFILES = {
    'baseline': {
        'epoch': 10, 'lr': 1e-5,
        'lambda_verifier': 0.3, 'lambda_knowledge': 0.0,
        'early_stop_start_epoch': 5, 'early_stop_patience': 4,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
    'run1': {
        'epoch': 10, 'lr': 1e-5,
        'lambda_verifier': 0.25, 'lambda_knowledge': 0.0,
        'early_stop_start_epoch': 5, 'early_stop_patience': 4,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
    'run2': {
        'epoch': 10, 'lr': 8e-6,
        'lambda_verifier': 0.25, 'lambda_knowledge': 0.0,
        'early_stop_start_epoch': 6, 'early_stop_patience': 5,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
    'run3': {
        'epoch': 10, 'lr': 8e-6,
        'lambda_verifier': 0.25, 'lambda_knowledge': 0.0,
        'early_stop_start_epoch': 6, 'early_stop_patience': 5,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
}

if RUN_PROFILE not in RUN_PROFILES:
    raise ValueError(f'Invalid RUN_PROFILE={RUN_PROFILE}')

RUN_TAG = f'{RUN_PROFILE}_{RUN_VARIANT}'
MODEL_FILENAME = f'best_model_gdinofrcnn_ncod_hum_{RUN_TAG}.ckpt'
LATEST_CKPT_FILENAME = f'latest_checkpoint_gdinofrcnn_ncod_hum_{RUN_TAG}.ckpt'
TRAIN_HISTORY_FILENAME = f'train_history_gdinofrcnn_ncod_hum_{RUN_TAG}.csv'
PREDICTIONS_CSV_FILENAME = f'test_predictions_gdinofrcnn_ncod_hum_{RUN_TAG}.csv'
METRICS_JSON_FILENAME = f'final_metrics_gdinofrcnn_ncod_hum_{RUN_TAG}.json'
BEST_ARTIFACT_NAME = f'best-model-gdinofrcnn-ncod-hum-{RUN_TAG}'
LAST_ARTIFACT_NAME = f'last-checkpoint-gdinofrcnn-ncod-hum-{RUN_TAG}'
FINAL_ARTIFACT_NAME = f'final-results-gdinofrcnn-ncod-hum-{RUN_TAG}'

FEAT_DIM = 1024  # DINOv3 frame
OBJ_DIM  = 2820  # FRCNN(2048) + DeBERTa cls(768) + bbox(4)
OBJ_BBOX_DIM = 4
print(f'\nBackbone: DINOv3 ({FEAT_DIM}-d frame) + GroundingDINO+FRCNN ({OBJ_DIM}-d obj, bbox split={OBJ_BBOX_DIM})')
print(f'Run profile: {RUN_PROFILE}')

class Config:
    # Paths
    video_feature_root = CLIP_FEATURE_PATH
    grounding_dino_path = GDINO_FEATURE_PATH
    sample_list_path = ANNOTATION_PATH
    split_dir_txt = SPLIT_DIR

    # Model architecture
    topK_frame = 16
    objs = 12
    frames = 16
    select_frames = 5
    topK_obj = 12

    frame_feat_dim = FEAT_DIM
    obj_feat_dim = OBJ_DIM
    use_grounding_dino = True

    # GDINO object encoding fixes
    obj_use_bbox_pos_embed = True
    obj_bbox_dim = OBJ_BBOX_DIM
    obj_hard_gather_from_frame = True
    obj_split_roi_class = True
    obj_roi_dim = 2048
    obj_class_dim = 768
    obj_mask_padding = True

    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3

    # Text encoder
    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    text_encoder_lr = 5e-6
    text_pool_mode = 1
    use_lora = False
    lora_r = 8
    lora_alpha = 16
    lora_dropout = 0.1
    lora_target_modules = ['query_proj', 'key_proj', 'value_proj']

    # Training
    bs = 32
    accumulation_steps = 1
    lr = 1e-5
    epoch = 10
    gpu = 0
    gamma = 0.5
    decay = 1e-4
    n_query = 5
    lambda_verifier = 0.3
    lambda_knowledge = 0.0
    return_family_id = True

    # LR scheduler + early stopping
    lr_schedule = 'cosine_warmup'
    warmup_epochs = 1
    lr_patience = 1
    min_lr = 1e-7
    early_stop_patience = 4
    early_stop_min_delta = 0.05
    early_stop_start_epoch = 5

    # Generic training improvements
    use_hard_neg_mining = True
    hard_neg_weight_max = 1.5
    use_ema = True
    ema_decay = 0.999

    # NCOD + HUM
    ncod_u_lr = 0.1
    hum_alpha = 1.0

    # Aux warmup
    aux_warmup_epochs = 2

    # Other
    hard_eval = True
    pos_ratio = 1.0
    neg_ratio = 1.0
    a = 1.0
    num_workers = 0

for _k, _v in RUN_PROFILES[RUN_PROFILE].items():
    setattr(Config, _k, _v)

if RUN_PROFILE == 'run3':
    if RUN3_REG_MODE == 'dropout':
        Config.dropout = 0.25
        Config.encoder_dropout = 0.25
    elif RUN3_REG_MODE == 'decay':
        Config.decay = 8e-5
    else:
        raise ValueError(f'Invalid RUN3_REG_MODE={RUN3_REG_MODE}')

args = Config()

if args.text_encoder_type != 'microsoft/deberta-base':
    raise ValueError('Train notebook uses DeBERTa v1 only.')

set_gpu_devices(args.gpu)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
print(f'Run tag: {RUN_TAG}')
print(f'Config: frame_feat_dim={args.frame_feat_dim}, obj_feat_dim={args.obj_feat_dim}, objs={args.objs}, select_frames={args.select_frames}')
print(f'use_grounding_dino={args.use_grounding_dino} -> obj_sorter SKIPPED')
print(f'obj_use_bbox_pos_embed={args.obj_use_bbox_pos_embed} | obj_hard_gather_from_frame={args.obj_hard_gather_from_frame}')
print(f'Effective bs: physical={args.bs} x accum={args.accumulation_steps} = {args.bs * args.accumulation_steps}')
print(f'lr={args.lr} | text_encoder_lr={args.text_encoder_lr} | decay={args.decay}')
print(f'use_lora={args.use_lora} | hard_neg={args.use_hard_neg_mining} (max={args.hard_neg_weight_max}) | ema={args.use_ema} (decay={args.ema_decay})')
print(f'lr_schedule={args.lr_schedule} | warmup_epochs={args.warmup_epochs}')
print(f'aux_warmup_epochs={args.aux_warmup_epochs} | verifier={args.lambda_verifier} | knowledge={args.lambda_knowledge}')
print(f'Early stop: start={args.early_stop_start_epoch}, patience={args.early_stop_patience}, min_delta={args.early_stop_min_delta}')
print(f'Model file: {MODEL_FILENAME}')

=== CELL 5: Paths & Config ===

--- Path Verification ---
OK DINOv3 Features (1024D): ['oIJ5MJTVBSw_000171_000181.pt', 'MswslLIhzyk_000100_000110.pt', '99wXNwpG4vY_000007_000017.pt']
OK GDINO+FRCNN Features (2820D): ['N3byaY-0E3E_000042_000052.pkl', '7RvfjAYqjcM_000224_000234.pkl', 'M7v6KZVI8hg_000123_000133.pkl']
OK Annotations (QA): ['8cGZHIlqRIc_000004_000014', 'u-IbgVFcshA_000000_000010', 'XGI0J0HCsEM_000045_000055']
OK Splits: ['valid.pkl', 'train.pkl', 'test.pkl']

DINOv3 .pt: 26900 | GDINO+FRCNN .pkl: 26900

Backbone: DINOv3 (1024-d frame) + GroundingDINO+FRCNN (2820-d obj, bbox split=4)
Run profile: run1

Device: cuda
Run tag: run1_selectivetopk_splitgate_mask_v2_nokb
Config: frame_feat_dim=1024, obj_feat_dim=2820, objs=12, select_frames=5
use_grounding_dino=True -> obj_sorter SKIPPED
obj_use_bbox_pos_embed=True | obj_hard_gather_from_frame=True
Effective bs: physical=32 x accum=1 = 32
lr=1e-05 | text_encoder_lr=5e-06 | decay=0.0001
use_lora=False | hard_neg=True (max=1.5) | em

In [ ]:
# CELL 6: Create validation and test datasets for inference
print('=== CELL 6: Validation/Test Datasets ===')
val_ds = VideoQADataset(
    split='val', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
test_ds = VideoQADataset(
    split='test', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
for split_name, dataset in [('val',val_ds),('test',test_ds)]:
    if len(dataset)==0: raise RuntimeError(f'{split_name} dataset is empty')
val_loader=DataLoader(val_ds,args.bs,shuffle=False,num_workers=0,pin_memory=False)
test_loader=DataLoader(test_ds,args.bs,shuffle=False,num_workers=0,pin_memory=False)
probe=test_ds[0][1]
if tuple(probe.shape)!=(args.topK_frame,args.objs,args.obj_feat_dim):
    raise RuntimeError(f'Object shape mismatch: {tuple(probe.shape)}')
print(f'Val: {len(val_ds)} | Test: {len(test_ds)} | object={tuple(probe.shape)}')


=== CELL 6: Validation/Test Datasets ===
[val] Object feature format: unknown
[val] Using GroundingDINO features
[val] Indexed 26900 object features, 26900 ViT features
[val] Loaded 2695 video IDs from /kaggle/input/casual-vid-data-split/split/valid.pkl
[val] ViT: 2695, Obj: 2695, Both: 2695


[val] Parsing annotations:   0%|          | 0/2695 [00:00<?, ?it/s]

[val] Final: 16170 QA pairs
[test] Object feature format: unknown
[test] Using GroundingDINO features
[test] Indexed 26900 object features, 26900 ViT features
[test] Loaded 5429 video IDs from /kaggle/input/casual-vid-data-split/split/test.pkl
[test] ViT: 5429, Obj: 5429, Both: 5429


[test] Parsing annotations:   0%|          | 0/5429 [00:00<?, ?it/s]

[test] Final: 32574 QA pairs
Val: 16170 | Test: 32574 | object=(16, 12, 2820)


In [ ]:
# CELL 7: Build inference model only
print('=== CELL 7: Inference Model ===')
cfg={k:v for k,v in Config.__dict__.items() if not k.startswith('_')}
cfg['device']=device
cfg['topK_frame']=args.select_frames
model=VideoQAmodel(**cfg).to(device)
model.eval()
model.hard_eval=True
save_path=os.path.join(MODEL_DIR,MODEL_FILENAME)
print(f'Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')
print(f'Expected artifact: {BEST_ARTIFACT_NAME}:best')


=== CELL 7: Inference Model ===


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] DebertaModel LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model parameters: 220.35M
Expected artifact: best-model-gdinofrcnn-ncod-hum-run1_selectivetopk_splitgate_mask_v2_nokb:best


In [ ]:
# CELL 8: Load best EMA checkpoint from W&B
print('=== CELL 8: Load Best W&B Checkpoint ===')
INFERENCE_RUN_TAG=f'inference_{RUN_TAG}'
run=wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY, name=INFERENCE_RUN_TAG,
    job_type='inference', config={'source_run_tag':RUN_TAG,'artifact':BEST_ARTIFACT_NAME},
    reinit='finish_previous'
)
artifact_ref=f'{WANDB_ENTITY}/{WANDB_PROJECT}/{BEST_ARTIFACT_NAME}:best'
print('Downloading:',artifact_ref)
artifact=wandb.Api().artifact(artifact_ref)
artifact_dir=artifact.download(root=os.path.join(BASE,'artifacts',BEST_ARTIFACT_NAME))
checkpoint_path=os.path.join(artifact_dir,MODEL_FILENAME)
if not os.path.exists(checkpoint_path):
    candidates=list(Path(artifact_dir).rglob('*.ckpt'))
    if not candidates: raise FileNotFoundError(f'No checkpoint in {artifact_dir}')
    checkpoint_path=str(candidates[0])
checkpoint=torch.load(checkpoint_path,map_location=device,weights_only=False)
state=checkpoint.get('ema_model_state_dict') or checkpoint.get('model_state_dict') or checkpoint
missing,unexpected=model.load_state_dict(state,strict=False)
if missing or unexpected:
    raise RuntimeError(f'Checkpoint architecture mismatch: missing={missing[:10]}, unexpected={unexpected[:10]}')
model.eval(); model.hard_eval=True
best_acc=float(checkpoint.get('best_acc',0.0))
best_epoch=int(checkpoint.get('best_epoch',checkpoint.get('epoch',0)))
print(f'Loaded best EMA weight: epoch={best_epoch}, val Acc_ALL={best_acc:.4f}%')


=== CELL 8: Load Best W&B Checkpoint ===


Downloading: haidang262004-i-h-c-qu-c-gia-tp-hcm/transtr-causalvid-dino/best-model-gdinofrcnn-ncod-hum-run1_selectivetopk_splitgate_mask_v2_nokb:best


wandb: Downloading large artifact 'best-model-gdinofrcnn-ncod-hum-run1_selectivetopk_splitgate_mask_v2_nokb:best', 3337.59MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:21.8 (152.9MB/s)


Loaded best EMA weight: epoch=4, val Acc_ALL=65.3061%


In [ ]:
# CELL 9: Deterministic inference smoke test
print('=== CELL 9: Deterministic Smoke Test ===')
batch=next(iter(val_loader))
ff,of,q,a,ans_id,keys,qfid=_unpack_batch(batch)
ff,of=ff.to(device),of.to(device)
qfid=qfid.to(device) if qfid is not None else None
with torch.inference_mode():
    out1=model(ff,of,q,a,return_aux=True,q_family_id=qfid)
    out2=model(ff,of,q,a,return_aux=True,q_family_id=qfid)
    pred1=out1.get('fused_score',out1['logits']).argmax(-1)
    pred2=out2.get('fused_score',out2['logits']).argmax(-1)
if not torch.equal(pred1,pred2): raise RuntimeError('hard_eval is not deterministic')
if not torch.isfinite(out1['logits']).all(): raise RuntimeError('Non-finite inference logits')
print('Deterministic predictions: OK | batch:',len(keys))
del batch,ff,of,q,a,ans_id,keys,qfid,out1,out2,pred1,pred2
torch.cuda.empty_cache()


=== CELL 9: Deterministic Smoke Test ===
Deterministic predictions: OK | batch: 32


In [ ]:
# CELL 10: Detailed Evaluation + CSV export (no knowledge bank / memory gating)
print('=== CELL 10: Detailed Evaluation (Model Only) ===')

CSV_OUTPUT_PATH = os.path.join(MODEL_DIR, PREDICTIONS_CSV_FILENAME)
COMPARISON_CSV_PATH = os.path.join(MODEL_DIR, 'run_comparison_gdino_3run.csv')

def _build_eval_meta_map(loader):
    sample_list = getattr(getattr(loader, 'dataset', None), 'sample_list', None)
    meta_map = {}
    if sample_list is None:
        return meta_map
    for _, row in sample_list.iterrows():
        vid = str(row.get('video_id', ''))
        qtype = str(row.get('type', 'unknown'))
        meta_map[f'{vid}_{qtype}'] = {
            'video_id': vid,
            'question_type': qtype,
            'question': str(row.get('question', '')),
            'answers': [str(row.get(f'a{i}', '')) for i in range(5)]
        }
    return meta_map

def evaluate_detailed_v2(model, loader, device, log_to_wandb=True):
    model.eval()
    type_results, prediction_rows = {}, []
    meta_map = _build_eval_meta_map(loader)
    with torch.no_grad():
        for batch in tqdm(loader):
            ff, of, qns, ans_word, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of = ff.to(device), of.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, qns, ans_word, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            preds, targets = probs.argmax(axis=-1), ans_id.numpy()

            for i, (key, pred, target, prob_vec) in enumerate(zip(qns_keys, preds, targets, probs)):
                meta = meta_map.get(str(key), {})
                qtype = str(meta.get('question_type', 'unknown'))
                video_id = str(meta.get('video_id', str(key)))
                answers = list(meta.get('answers', ['', '', '', '', '']))[:5]
                answers += [''] * (5 - len(answers))
                correct_idx, predicted_idx = int(target), int(pred)
                is_correct = int(correct_idx == predicted_idx)
                type_results.setdefault(qtype, []).append({
                    'video_id': video_id, 'pred': predicted_idx,
                    'target': correct_idx, 'correct': bool(is_correct)
                })
                row = {
                    'video_id': video_id, 'question_type': qtype,
                    'question': str(meta.get('question', qns[i])),
                    'correct_idx': correct_idx, 'predicted_idx': predicted_idx,
                    'is_correct': is_correct,
                    'confidence': float(prob_vec[predicted_idx]),
                    'predicted_answer': answers[predicted_idx],
                    'correct_answer': answers[correct_idx]
                }
                for j in range(5):
                    row[f'a{j}'] = answers[j]
                    row[f'prob_a{j}'] = float(prob_vec[j])
                prediction_rows.append(row)

    if not prediction_rows:
        raise RuntimeError(
            'Detailed evaluation produced zero predictions. Check that test_ds/test_loader is non-empty.'
        )
    prediction_df = pd.DataFrame(prediction_rows)
    prediction_df.to_csv(CSV_OUTPUT_PATH, index=False)

    # Full-test calibration diagnostics.
    prob_cols = [f'prob_a{i}' for i in range(5)]
    all_probs = prediction_df[prob_cols].to_numpy(dtype=np.float64)
    all_targets = prediction_df['correct_idx'].to_numpy(dtype=np.int64)
    all_conf = all_probs.max(axis=1)
    all_pred = all_probs.argmax(axis=1)
    all_correct = (all_pred == all_targets).astype(np.float64)
    gt_prob = all_probs[np.arange(len(all_targets)), all_targets]
    calibration = {
        'NLL': float(-np.log(np.clip(gt_prob, 1e-12, 1.0)).mean()),
        'Brier': float(np.square(all_probs - np.eye(5)[all_targets]).sum(axis=1).mean()),
        'Mean_Confidence': float(all_conf.mean()),
        'Mean_Confidence_Correct': float(all_conf[all_correct == 1].mean()) if np.any(all_correct == 1) else 0.0,
        'Mean_Confidence_Wrong': float(all_conf[all_correct == 0].mean()) if np.any(all_correct == 0) else 0.0,
        'Wrong_Confidence_Above_0_9': float(np.mean(all_conf[all_correct == 0] >= 0.9) * 100) if np.any(all_correct == 0) else 0.0,
    }
    ece = 0.0
    calibration_rows = []
    for low in np.linspace(0.0, 1.0, 16)[:-1]:
        high = low + 1.0 / 15.0
        in_bin = (all_conf > low) & (all_conf <= high if high < 1.0 else all_conf <= 1.0)
        if not np.any(in_bin):
            continue
        bin_acc = float(all_correct[in_bin].mean())
        bin_conf = float(all_conf[in_bin].mean())
        weight = float(in_bin.mean())
        ece += weight * abs(bin_acc - bin_conf)
        calibration_rows.append({'bin_low': low, 'bin_high': high, 'count': int(in_bin.sum()), 'accuracy': bin_acc, 'confidence': bin_conf})
    calibration['ECE_15'] = float(ece)
    CALIBRATION_CSV_PATH = os.path.join(MODEL_DIR, f'calibration_bins_{RUN_TAG}.csv')
    pd.DataFrame(calibration_rows).to_csv(CALIBRATION_CSV_PATH, index=False)
    print(f'Saved detailed predictions CSV: {CSV_OUTPUT_PATH}')

    metrics = {}
    mapping = [
        ('Description', 'descriptive'), ('Explanation', 'explanatory'),
        ('Predictive-Answer', 'predictive'), ('Predictive-Reason', 'predictive_reason'),
        ('Counterfactual-Answer', 'counterfactual'),
        ('Counterfactual-Reason', 'counterfactual_reason')
    ]
    for name, qtype in mapping:
        rows = type_results.get(qtype, [])
        metrics[name] = 100.0 * sum(r['correct'] for r in rows) / max(len(rows), 1)

    def _paired_metric(ans_type, reason_type):
        ans = {r['video_id']: r['correct'] for r in type_results.get(ans_type, [])}
        reason = {r['video_id']: r['correct'] for r in type_results.get(reason_type, [])}
        common = set(ans) & set(reason)
        return 100.0 * sum(ans[v] and reason[v] for v in common) / max(len(common), 1)

    metrics['PAR'] = _paired_metric('predictive', 'predictive_reason')
    metrics['CAR'] = _paired_metric('counterfactual', 'counterfactual_reason')
    metrics['Acc_ALL'] = (metrics['Description'] + metrics['Explanation'] + metrics['PAR'] + metrics['CAR']) / 4.0
    metrics.update({f'Calibration_{k}': v for k, v in calibration.items()})

    metrics['WeightedScore_WeakPriority'] = (
        0.35 * metrics['Predictive-Reason'] +
        0.35 * metrics['Counterfactual-Reason'] +
        0.20 * metrics['Acc_ALL'] + 0.10 * float(best_acc)
    )
    if log_to_wandb and wandb.run is not None:
        wandb.log({f'eval/{k.replace("-", "_")}': float(v) for k, v in metrics.items()})
    print(f"PAR: {metrics['PAR']:.2f}% | CAR: {metrics['CAR']:.2f}% | Acc_ALL: {metrics['Acc_ALL']:.2f}%")
    print('Calibration:', {k: round(v, 4) for k, v in calibration.items()})
    return metrics, type_results, CSV_OUTPUT_PATH

metrics, raw_results, predictions_csv = evaluate_detailed_v2(model, test_loader, device, log_to_wandb=True)
comparison_row = {
    'run_tag': RUN_TAG, 'run_profile': RUN_PROFILE,
    'best_val_acc': float(best_acc), 'best_epoch': int(best_epoch),
    **{k: float(v) for k, v in metrics.items()}
}
if os.path.exists(COMPARISON_CSV_PATH):
    comp_df = pd.read_csv(COMPARISON_CSV_PATH)
    comp_df = comp_df[comp_df['run_tag'] != RUN_TAG]
    comp_df = pd.concat([comp_df, pd.DataFrame([comparison_row])], ignore_index=True)
else:
    comp_df = pd.DataFrame([comparison_row])
comp_df = comp_df.sort_values('WeightedScore_WeakPriority', ascending=False)
comp_df.to_csv(COMPARISON_CSV_PATH, index=False)
display(comp_df[['run_tag', 'best_val_acc', 'Predictive-Reason', 'Counterfactual-Reason', 'Acc_ALL', 'WeightedScore_WeakPriority']])
if wandb.run is not None:
    wandb.run.summary.update({
        'run_tag': RUN_TAG, 'run_profile': RUN_PROFILE,
        'weighted_score_weak_priority': float(metrics['WeightedScore_WeakPriority'])
    })
print(metrics)


=== CELL 10: Detailed Evaluation (Model Only) ===


  0%|          | 0/1018 [00:00<?, ?it/s]

Saved detailed predictions CSV: /content/working/models/test_predictions_gdinofrcnn_ncod_hum_run1_selectivetopk_splitgate_mask_v2_nokb.csv
PAR: 52.13% | CAR: 51.85% | Acc_ALL: 64.39%
Calibration: {'NLL': 0.9679, 'Brier': 0.4416, 'Mean_Confidence': 0.8155, 'Mean_Confidence_Correct': 0.8593, 'Mean_Confidence_Wrong': 0.7087, 'Wrong_Confidence_Above_0_9': 26.3992, 'ECE_15': 0.1062}


,run_tag,best_val_acc,Predictive-Reason,Counterfactual-Reason,Acc_ALL,WeightedScore_WeakPriority
0,run1_selectivetopk_splitgate_mask_v2_nokb,65.306122,67.470989,66.181617,64.394916,66.188008


{'Description': 74.50727574138884, 'Explanation': 79.09375575612452, 'Predictive-Answer': 68.1525142751888, 'Predictive-Reason': 67.47098913243691, 'Counterfactual-Answer': 70.16025050653896, 'Counterfactual-Reason': 66.18161724074415, 'PAR': 52.12746362129305, 'CAR': 51.85116964450175, 'Acc_ALL': 64.39491619082705, 'Calibration_NLL': 0.9678822244230377, 'Calibration_Brier': 0.4416417801209397, 'Calibration_Mean_Confidence': 0.8155128210023629, 'Calibration_Mean_Confidence_Correct': 0.8593111129456982, 'Calibration_Mean_Confidence_Wrong': 0.7086579385251909, 'Calibration_Wrong_Confidence_Above_0_9': 26.399155227032733, 'Calibration_ECE_15': 0.10623548324832591, 'WeightedScore_WeakPriority': 66.18800771367674}


In [ ]:
# CELL 10B: Object, selector, and global-frame diagnostics (single-pass)
print('=== CELL 10B: Object + Selector + Global Frame Ablations ===')
ROI_DIM, CLASS_DIM, BBOX_DIM = 2048, 768, 4

def _branch_statistics(model, loader, device, max_batches=50):
    sums = {k: 0.0 for k in ['roi_input_l2','class_input_l2','bbox_input_l2',
        'roi_projected_l2','class_projected_l2','bbox_projected_l2','valid_object_ratio']}
    count = 0
    model.eval()
    with torch.inference_mode():
        for bi, batch in enumerate(tqdm(loader, desc='Object diagnostics')):
            if bi >= max_batches: break
            _, of, *_ = _unpack_batch(batch)
            of = of.to(device)
            roi, cls, bbox = of[..., :ROI_DIM], of[..., ROI_DIM:ROI_DIM+CLASS_DIM], of[..., -BBOX_DIM:]
            valid = model._object_valid_mask(of)
            if not valid.any(): continue
            rv, cv, bv = roi[valid], cls[valid], bbox[valid]
            rp = model.obj_roi_resize(model.obj_roi_norm(rv))
            cp = model.obj_class_resize(model.obj_class_norm(cv))
            bp = model.bbox_pos_embed(bv)
            for key, value in [('roi_input_l2',rv),('class_input_l2',cv),('bbox_input_l2',bv),
                               ('roi_projected_l2',rp),('class_projected_l2',cp),('bbox_projected_l2',bp)]:
                sums[key] += value.norm(dim=-1).mean().item()
            sums['valid_object_ratio'] += valid.float().mean().item()
            count += 1
    stats = {k:v/max(count,1) for k,v in sums.items()}
    gates = (2.0 * torch.sigmoid(model.obj_gate_logits.detach())).cpu().tolist()
    stats.update({'gate_roi':gates[0], 'gate_class':gates[1], 'gate_bbox':gates[2]})
    stats['gated_roi_to_class_ratio'] = (gates[0]*stats['roi_projected_l2']) / max(gates[1]*stats['class_projected_l2'],1e-8)
    return stats

def _apply_object_ablation(of, mode):
    x=of.clone()
    if mode=='normal': return x
    if mode=='no_roi': x[...,:ROI_DIM]=0
    elif mode=='no_class': x[...,ROI_DIM:ROI_DIM+CLASS_DIM]=0
    elif mode=='no_bbox': x[...,-BBOX_DIM:]=0
    elif mode=='roi_only': x[...,ROI_DIM:]=0
    elif mode=='class_only': x[...,:ROI_DIM]=0; x[...,-BBOX_DIM:]=0
    elif mode=='zero_object': x.zero_()
    else: raise ValueError(mode)
    return x

def _new_accumulator():
    return {'correct':0, 'total':0, 'type_results':{}}

def _update_accumulator(acc, keys, pred, tgt):
    acc['correct'] += (pred == tgt).sum().item()
    acc['total'] += tgt.numel()
    for key, pred_i, target_i in zip(keys, pred.cpu().tolist(), tgt.cpu().tolist()):
        vid, qtype = _split_qns_key(key)
        acc['type_results'].setdefault(qtype, []).append({
            'video_id':vid, 'correct':bool(pred_i == target_i)
        })

def _finish_accumulator(acc):
    m = _compute_acc_all_metrics(acc['type_results'])
    m['Plain_Acc'] = 100.0 * acc['correct'] / max(acc['total'], 1)
    return m

# One pass through validation. The raw DeBERTa question/answer embeddings are
# cached only for the current batch and reused by every ablation mode. This is
# exact in model.eval() and avoids encoding the same text 12 times.
specs = [
    {'name':'normal',         'object_mode':'normal',      'frame_mode':'normal',         'selector':None},
    {'name':'no_roi',         'object_mode':'no_roi',      'frame_mode':'normal',         'selector':None},
    {'name':'no_class',       'object_mode':'no_class',    'frame_mode':'normal',         'selector':None},
    {'name':'no_bbox',        'object_mode':'no_bbox',     'frame_mode':'normal',         'selector':None},
    {'name':'roi_only',       'object_mode':'roi_only',    'frame_mode':'normal',         'selector':None},
    {'name':'class_only',     'object_mode':'class_only',  'frame_mode':'normal',         'selector':None},
    {'name':'zero_object',    'object_mode':'zero_object', 'frame_mode':'normal',         'selector':None},
    {'name':'uniform',        'object_mode':'normal',      'frame_mode':'normal',         'selector':'uniform'},
    {'name':'random',         'object_mode':'normal',      'frame_mode':'normal',         'selector':'random'},
    {'name':'zero_frame',     'object_mode':'normal',      'frame_mode':'zero_frame',     'selector':None},
    {'name':'shuffled_frame', 'object_mode':'normal',      'frame_mode':'shuffled_frame', 'selector':None},
]
accumulators = {spec['name']:_new_accumulator() for spec in specs}
branch_stats = _branch_statistics(model, val_loader, device)
print(pd.Series(branch_stats).round(4).to_string())

model.eval()
old_hard, old_override = model.hard_eval, model.frame_selection_override
model.hard_eval = True
try:
    with torch.inference_mode():
        for batch in tqdm(val_loader, desc='Single-pass full ablation suite'):
            ff, of, q, a, ans_id, keys, qfid = _unpack_batch(batch)
            ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
            qfid = qfid.to(device) if qfid is not None else None

            original_forward_text = model.forward_text
            text_cache = {}
            def cached_forward_text(text_queries, text_device, has_ans=False):
                cache_key = ('answer' if has_ans else 'question', tuple(text_queries))
                if cache_key not in text_cache:
                    text_cache[cache_key] = original_forward_text(text_queries, text_device, has_ans=has_ans)
                return text_cache[cache_key]
            model.forward_text = cached_forward_text
            try:
                for spec in specs:
                    model.frame_selection_override = spec['selector']
                    ff_mode = ff
                    if spec['frame_mode'] == 'zero_frame':
                        ff_mode = torch.zeros_like(ff)
                    elif spec['frame_mode'] == 'shuffled_frame' and ff.size(0) > 1:
                        ff_mode = ff.roll(1, 0)
                    of_mode = _apply_object_ablation(of, spec['object_mode'])
                    out = model(ff_mode, of_mode, q, a, return_aux=True, q_family_id=qfid)
                    pred = out.get('fused_score', out['logits']).argmax(-1)
                    _update_accumulator(accumulators[spec['name']], keys, pred, tgt)
            finally:
                model.forward_text = original_forward_text
                text_cache.clear()
finally:
    model.hard_eval, model.frame_selection_override = old_hard, old_override

all_rows = [{'mode':spec['name'], **_finish_accumulator(accumulators[spec['name']])} for spec in specs]
all_df = pd.DataFrame(all_rows)
normal_acc = float(all_df.loc[all_df['mode']=='normal','Acc_ALL'].iloc[0])
all_df['drop_vs_normal_Acc_ALL'] = normal_acc - all_df['Acc_ALL']

object_modes = ['normal','no_roi','no_class','no_bbox','roi_only','class_only','zero_object']
object_ablation_df = all_df[all_df['mode'].isin(object_modes)].reset_index(drop=True)
selector_modes = ['normal','uniform','random','zero_frame','shuffled_frame']
selector_frame_df = all_df[all_df['mode'].isin(selector_modes)].reset_index(drop=True)
selector_frame_df = selector_frame_df.rename(columns={'drop_vs_normal_Acc_ALL':'drop_vs_learned_Acc_ALL'})

OBJECT_ABLATION_CSV_PATH=os.path.join(MODEL_DIR,f'object_branch_ablation_{RUN_TAG}.csv')
SELECTOR_FRAME_CSV_PATH=os.path.join(MODEL_DIR,f'selector_global_frame_ablation_{RUN_TAG}.csv')
object_ablation_df.to_csv(OBJECT_ABLATION_CSV_PATH,index=False)
selector_frame_df.to_csv(SELECTOR_FRAME_CSV_PATH,index=False)
display(object_ablation_df[['mode','Acc_ALL','Plain_Acc','PAR','CAR','drop_vs_normal_Acc_ALL']])
display(selector_frame_df[['mode','Acc_ALL','Plain_Acc','PAR','CAR','drop_vs_learned_Acc_ALL']])
print('Saved:',OBJECT_ABLATION_CSV_PATH,SELECTOR_FRAME_CSV_PATH)
if wandb.run is not None:
    wandb.log({f'object_diagnostics/{k}':float(v) for k,v in branch_stats.items()})


=== CELL 10B: Object + Selector + Global Frame Ablations ===


Object diagnostics:   0%|          | 0/506 [00:00<?, ?it/s]

roi_input_l2                18.0571
class_input_l2              19.7180
bbox_input_l2                1.1611
roi_projected_l2            27.7547
class_projected_l2          27.6770
bbox_projected_l2           19.5959
valid_object_ratio           0.5099
gate_roi                     1.0072
gate_class                   0.9929
gate_bbox                    0.2384
gated_roi_to_class_ratio     1.0172


Single-pass full ablation suite:   0%|          | 0/506 [00:00<?, ?it/s]

,mode,Acc_ALL,Plain_Acc,PAR,CAR,drop_vs_normal_Acc_ALL
0,normal,65.306122,71.193568,52.319109,53.580705,0.000000
1,no_roi,64.777365,70.810142,50.946197,53.358071,0.528757
2,no_class,64.916512,70.865801,51.243043,53.729128,0.389610
3,no_bbox,65.324675,71.181200,52.319109,53.654917,-0.018553
4,roi_only,64.897959,70.841064,51.317254,53.692022,0.408163
5,class_only,64.768089,70.828695,50.983302,53.320965,0.538033
6,zero_object,64.814471,70.884354,51.094620,53.766234,0.491651


,mode,Acc_ALL,Plain_Acc,PAR,CAR,drop_vs_learned_Acc_ALL
0,normal,65.306122,71.193568,52.319109,53.580705,0.000000
1,uniform,65.343228,71.280148,52.504638,53.766234,-0.037106
2,random,65.361781,71.218306,52.504638,53.654917,-0.055659
3,zero_frame,52.263451,58.268398,25.528757,45.788497,13.042672
4,shuffled_frame,62.022263,68.911565,50.723562,53.098330,3.283859


Saved: /content/working/models/object_branch_ablation_run1_selectivetopk_splitgate_mask_v2_nokb.csv /content/working/models/selector_global_frame_ablation_run1_selectivetopk_splitgate_mask_v2_nokb.csv


In [ ]:
# CELL 10C: Balanced qualitative analysis of 20 wrong predictions
print('=== CELL 10C: Twenty Wrong Prediction Examples ===')
if not os.path.exists(predictions_csv) or os.path.getsize(predictions_csv) == 0:
    raise RuntimeError('Prediction CSV is missing or empty; run CELL 10 successfully first.')
prediction_df = pd.read_csv(predictions_csv)
if prediction_df.empty:
    raise RuntimeError('Prediction CSV has no rows; check test dataset resolution.')
wrong_df = prediction_df[prediction_df['is_correct'] == 0].copy()
if wrong_df.empty:
    print('No wrong predictions found; skipping qualitative error samples.')
wrong_df['gt_probability'] = wrong_df.apply(
    lambda r: float(r.get(f"prob_a{int(r['correct_idx'])}", 0.0)), axis=1
)
wrong_df['wrong_margin'] = wrong_df['confidence'] - wrong_df['gt_probability']
wrong_df = wrong_df.sort_values(['question_type', 'wrong_margin'], ascending=[True, False])

# Round-robin selection guarantees broad question-type coverage before adding extras.
groups = {q: g.reset_index(drop=True) for q, g in wrong_df.groupby('question_type')}
selected = []
depth = 0
while len(selected) < 20 and any(depth < len(g) for g in groups.values()):
    for qtype in sorted(groups):
        if depth < len(groups[qtype]) and len(selected) < 20:
            selected.append(groups[qtype].iloc[depth])
    depth += 1

error_samples_df = pd.DataFrame(selected)
ERROR_SAMPLES_CSV_PATH = os.path.join(MODEL_DIR, f'error_samples_20_{RUN_TAG}.csv')
error_samples_df.to_csv(ERROR_SAMPLES_CSV_PATH, index=False)

show_cols = [
    'video_id', 'question_type', 'question', 'correct_answer',
    'predicted_answer', 'confidence', 'gt_probability', 'wrong_margin'
]
display(error_samples_df[show_cols].style.format({
    'confidence': '{:.3f}', 'gt_probability': '{:.3f}', 'wrong_margin': '{:.3f}'
}))
for i, row in error_samples_df.reset_index(drop=True).iterrows():
    candidates = ' | '.join([f"{j}:{row[f'a{j}']} ({row[f'prob_a{j}']:.3f})" for j in range(5)])
    print(f"\n[{i+1}] {row['question_type']} | video={row['video_id']}")
    print('Q:', row['question'])
    print('GT:', row['correct_answer'])
    print('Pred:', row['predicted_answer'])
    print('Candidates:', candidates)
print(f'\nSaved: {ERROR_SAMPLES_CSV_PATH}')


=== CELL 10C: Twenty Wrong Prediction Examples ===


,video_id,question_type,question,correct_answer,predicted_answer,confidence,gt_probability,wrong_margin
0,UdbKwNh_5lw_000066_000076,counterfactual,What will happen if [person_1] is sitting outdoors?,[person_1] won't be able to continue practicing.,[person_1] will not be able to play music outdoors.,0.996,0.001,0.995
0,QjEShHP7RPg_000149_000159,counterfactual_reason,What will happen if [person_1] finishes performing the solo?,[person_1] is helpful.,[person_1] will not need to hold the bass guitar after playing music.,0.995,0.002,0.994
0,Xllal9867PY_000015_000025,descriptive,What is this place?,The room is colorful.,It looks like a fashion shop.,0.999,0.000,0.998
0,sOHBUjoJddg_000044_000054,explanatory,Why is [person_2] in the room?,[person_2] is practicing roller skating.,Because [person_2] needs a big room to play musical instruments with friends.,0.998,0.001,0.997
0,XI139xM0YJk_000000_000010,predictive,What is [person_1] going to do next?,[person_1] will continue standing here to see them practicing.,[person_1] is going to wait for [person_2] to help him tie a bow.,0.999,0.000,0.999
0,1DAtchstb_4_000004_000014,predictive_reason,What is [person_2] going to do?,Maybe [person_2] can't keep balance for long.,The gift box was lying on the ground.,0.999,0.000,0.998
1,4mqM4BgSqPo_000026_000036,counterfactual,What would happen if [person_1] did not wear glasses?,[person_1] will stop to have a rest.,[person_1] could have difficulty in playing accordion.,0.995,0.001,0.994
1,81RP4GDLBAc_000004_000014,counterfactual_reason,What if [person_1] was not careful when the hammer leans to his mouth?,A tool is better than hands only.,The hammer is heavy and [person_1] is adventuring.,0.995,0.002,0.993
1,_KapWUYOm64_000040_000050,descriptive,Where is [person_2]?,[person_2] is in the swimming pool.,[person_2] is sitting on [bed_2].,0.997,0.001,0.996
1,CkZeAJUOF0o_000016_000026,explanatory,Why do [person_1] and [person_2] make a fire?,[person_1] wants to polish the shoe.,To make the log cabin warmer.,0.997,0.001,0.996



[1] counterfactual | video=UdbKwNh_5lw_000066_000076
Q: What will happen if [person_1] is sitting outdoors?
GT: [person_1] won't be able to continue practicing.
Pred: [person_1] will not be able to play music outdoors.
Candidates: 0:[person_1] won't be able to continue practicing. (0.001) | 1:[person_1]'s hair won't float in a messy. (0.001) | 2:[person_1] will stay at home. (0.001) | 3:[person_1] will not be able to play music outdoors. (0.996) | 4:[person_1] will take a shower. (0.001)

[2] counterfactual_reason | video=QjEShHP7RPg_000149_000159
Q: What will happen if [person_1] finishes performing the solo?
GT: [person_1] is helpful.
Pred: [person_1] will not need to hold the bass guitar after playing music.
Candidates: 0:[person_1] will not need to hold the bass guitar after playing music. (0.995) | 1:Without the frisbee , they can not play the game. (0.001) | 2:[person_1] is helpful. (0.002) | 3:Because [person_1] falls when she has not finished her turn. (0.001) | 4:The card box

In [ ]:
# CELL 11: Finish inference W&B run (metrics only, no artifacts)
print('=== CELL 11: Finish Inference W&B ===')
METRICS_JSON_PATH=os.path.join(MODEL_DIR,f'inference_{METRICS_JSON_FILENAME}')
with open(METRICS_JSON_PATH,'w') as f: json.dump(metrics,f,indent=2)
if wandb.run is not None:
    wandb.run.summary.update({
        'source_best_epoch':best_epoch,
        'source_best_val_acc':best_acc,
        'test_Acc_ALL':float(metrics['Acc_ALL']),
        'test_PAR':float(metrics['PAR']),
        'test_CAR':float(metrics['CAR'])
    })
    wandb.finish()
print('Inference complete. No artifact was uploaded.')


=== CELL 11: Finish Inference W&B ===


eval/Acc_ALL,▁
eval/CAR,▁
eval/Calibration_Brier,▁
eval/Calibration_ECE_15,▁
eval/Calibration_Mean_Confidence,▁
eval/Calibration_Mean_Confidence_Correct,▁
eval/Calibration_Mean_Confidence_Wrong,▁
eval/Calibration_NLL,▁
eval/Calibration_Wrong_Confidence_Above_0_9,▁
eval/Counterfactual_Answer,▁
+18,...


Inference complete. No artifact was uploaded.


In [ ]:
# # CELL 12: Disconnect Colab runtime after uploads
# print('Training and W&B uploads complete. Disconnecting runtime...')
# try:
#     from google.colab import runtime
#     runtime.unassign()
# except Exception as exc:
#     print(f'Automatic disconnect unavailable: {exc}')
